In [1]:
!nvidia-smi

Mon Aug 23 22:14:33 2021       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.57.02    Driver Version: 471.41       CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  Off  | 00000000:4B:00.0  On |                  N/A |
|  0%   37C    P8    37W / 300W |   1728MiB / 11264MiB |    ERR!      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
!nvcc --version

/bin/bash: nvcc: command not found


In [3]:
import os
import random
from datetime import datetime
import time
import itertools
import concurrent
from concurrent.futures import ThreadPoolExecutor
import cshogi.cli

In [4]:
os.chdir('..')
# os.chdir('..')

In [5]:
os.getcwd()

'/mnt/c/Users/hmats/workspace/DeepLearningShogi'

In [6]:
! pip install -e .

Obtaining file:///mnt/c/Users/hmats/workspace/DeepLearningShogi
  Attempting uninstall: dlshogi
    Found existing installation: dlshogi 0.0.6.1
    Uninstalling dlshogi-0.0.6.1:
      Successfully uninstalled dlshogi-0.0.6.1
  Running setup.py develop for dlshogi
You should consider upgrading via the '/home/hmatsuya/anaconda3/bin/python -m pip install --upgrade pip' command.


In [7]:
# from dlshogi.fbs_models.policy_value_network_resnet10 import PolicyValueNetwork
# from dlshogi.fbs_models import PolicyValueNetwork
# test = PolicyValueNetwork()

In [8]:
# os.listdir('//corgi/hmatsuya/workspace/Shogi/data/dl_data/hcpe')
# os.listdir('/mnt/corgi/hmatsuya/workspace/Shogi/data/dl_data/hcpe')

In [9]:
os.getcwd()

'/mnt/c/Users/hmats/workspace/DeepLearningShogi'

In [10]:
os.chdir('dlshogi')

In [11]:
#os.mkdir('../model')

In [12]:
#os.mkdir('../log')

## Test Feature Boosting and Supression

In [ ]:
# advantage: no value
# val_lambda: 0
# tanh
# polic_coef: 0.5

chunk_size = 38 * 20 * 1000 * 1000
activation='C:\\Anaconda3\\Scripts\\activate.bat'
test='/mnt/corgi/hmatsuya/workspace/Shogi/data/dl_data/hcpe/floodgate_teacher_uniq-test-01'

network="dlshogi.fbs_models.policy_value_network_resnet10.PolicyValueNetwork"

#! wsl ssh hmatsuya@corgi ls -lh /home/hmatsuya/workspace/Shogi/data/apery_teacher/

# resume = '-r ../model/state-2019'
resume = ''
teacher_dir = '/mnt/corgi/hmatsuya/workspace/Shogi/data/apery_teacher/shuffled'
filelist = os.listdir(teacher_dir)
hcpelist = list(filter(lambda f: '.hcpe' in f, filelist))
files = []
# project = "test_critic"
project = "test_critic"
test_name = "fbs10-result_critic"
name = f'{test_name}'
run_id = f'{name}.{time.time()}'
i = 0
if i == 0:
    resume = ''
else:
    resume = f'-m ../model/model-{name}-{i-1} -r ../model/checkpoint-{name}-{i-1}.pth'

for file in random.sample(hcpelist, len(hcpelist)):
    
    files.append(file)
    if len(files) < 10:
        continue
    
    teacher=' '.join([f'{teacher_dir}/{f}' for f in files])
    
    model=f'../model/model-{name}-{i}'
    checkpoint=f'../model/checkpoint-{name}-{i}.pth'
    log=f'../log/{name}.txt'
    ! python -u train.py {teacher} {test} --network {network} --use_result_critic --model {model} --checkpoint {checkpoint} {resume} --val_lambda 0.3 --lr 0.001 --weight_decay 0.00001 --use_average --use_evalfix --use_swa --log {log} --use_amp --project {project} --run_id {run_id}

    files.clear()
    resume = f'-r {checkpoint} -m {model}'
    i += 1


2021/08/23 22:14:52	INFO	network dlshogi.fbs_models.policy_value_network_resnet10.PolicyValueNetwork
2021/08/23 22:14:52	INFO	batchsize=1024
2021/08/23 22:14:52	INFO	lr=0.001
2021/08/23 22:14:52	INFO	weight_decay=1e-05
2021/08/23 22:14:52	INFO	val_lambda=0.3
wandb: Currently logged in as: hmatsuya (use `wandb login --relogin` to force relogin)
wandb: wandb version 0.12.0 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.11.2
wandb: Syncing run fbs10-result_critic
wandb:  View project at https://wandb.ai/hmatsuya/test_critic
wandb:  View run at https://wandb.ai/hmatsuya/test_critic/runs/fbs10-result_critic.1629724490.3422816
wandb: Run data is saved locally in /mnt/c/Users/hmats/workspace/DeepLearningShogi/dlshogi/wandb/run-20210823_221503-fbs10-result_critic.1629724490.3422816
wandb: Run `wandb offline` to turn off syncing.

2021/08/23 22:15:10	INFO	use swa(swa_start_epoch=1, swa_freq=250, swa_n_avr=10)
2021/08/23 22:1

In [ ]:
os.getcwd()

'/mnt/c/Users/hmats/workspace/DeepLearningShogi/notebook'

In [3]:
# Convert to ONNX
!python ../dlshogi/convert_model_to_onnx.py --network dlshogi.fbs_models.policy_value_network_resnet10.PolicyValueNetwork ../model/model-fbs10-result_critic-15 ../model/model-fbs10-result_critic-15.onnx
! scp ../model/model-fbs10-result_critic-15.onnx corgi:/home/hmatsuya/workspace/Shogi/dlcobra/model/

graph(%input1 : Float(1:5022, 62:81, 9:9, 9:1, requires_grad=0, device=cuda:0),
      %input2 : Float(1:4617, 57:81, 9:9, 9:1, requires_grad=0, device=cuda:0),
      %l1_1_1.weight : Float(192:558, 62:9, 3:3, 3:1, requires_grad=1, device=cuda:0),
      %l1_1_2.weight : Float(192:62, 62:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l1_2.weight : Float(192:57, 57:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l22.weight : Float(27:192, 192:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l22_2.bias : Float(2187:1, requires_grad=1, device=cuda:0),
      %l23_v.weight : Float(256:2187, 2187:1, requires_grad=1, device=cuda:0),
      %l23_v.bias : Float(256:1, requires_grad=1, device=cuda:0),
      %l24_v.weight : Float(1:256, 256:1, requires_grad=1, device=cuda:0),
      %l24_v.bias : Float(1:1, requires_grad=1, device=cuda:0),
      %norm1.weight : Float(192:1, requires_grad=1, device=cuda:0),
      %norm1.bias : Float(192:1, requires_grad=1, device=cuda:0),
      %norm1.r